# Precision & Recall: LLM Content Moderation

This notebook walks through how to evaluate an LLM-based classifier using **precision** and **recall** — two of the most important metrics for content moderation and classification tasks.

## The Dataset
We're working with app store reviews that have been **manually labeled** by humans for three types of harmful content:
- `racism`
- `bullying`  
- `sexual`

Reviews that have been labeled (value is `0` or `1`) form the **golden set** — our ground truth.

## The Goal
We'll build an LLM-based classifier that predicts whether a review contains unwanted sexual content, then measure how well it performs against the human labels.

---

## Key Concepts

**Precision** answers: *"Of everything the model flagged, what fraction was actually harmful?"*
- High precision = few false alarms
- `precision = true_positives / (true_positives + false_positives)`

**Recall** answers: *"Of all the actually harmful content, what fraction did the model catch?"*
- High recall = few misses
- `recall = true_positives / (true_positives + false_negatives)`

There's usually a trade-off: tuning your prompt to catch more bad content (higher recall) tends to also flag more innocent content (lower precision), and vice versa. Understanding this trade-off is the whole point of this exercise.


---
## Step 1: Read the Data

In [1]:
import pandas as pd

df = pd.read_csv("reviews-actual.csv", low_memory=False)

print(f"Total rows: {len(df)}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(10)


Total rows: 131749

Column names: ['reviewId', 'app', 'date', 'rating', 'userId', 'userName', 'title', 'body', 'sexually_explicit', 'unwanted_sexual', 'racism', 'bullying', 'spam']

First few rows:


,reviewId,app,date,rating,userId,userName,title,body,sexually_explicit,unwanted_sexual,racism,bullying,spam
0,5177791428,Monkey,2019-11-21T14:04:28-05:00,5,1026400969,Galrax171044085,Like it but it could be better,I love this app but y’all really really need t...,NaN,NaN,NaN,NaN,NaN
1,5177389656,Monkey,2019-11-21T11:40:29-05:00,4,1102399827,fidel escobar,Hi,Good,NaN,NaN,NaN,NaN,NaN
2,5177178523,Monkey,2019-11-21T10:33:40-05:00,5,1035192891,Jing_Yaboi,Good,Ya good,NaN,NaN,NaN,NaN,NaN
3,5176915921,Holla,2019-11-21T09:19:16-05:00,1,24431599,Michael in Dubai,Rip off,This app charges for everything now and is con...,NaN,NaN,NaN,NaN,NaN
4,5176416573,Yubo,2019-11-21T07:00:51-05:00,5,554354238,AM Official,Getting annoyed,Okay so as someone who does online schooling I...,NaN,NaN,NaN,NaN,NaN
5,5176328843,Monkey,2019-11-21T06:33:11-05:00,5,485829199,betbetbetbetbet6666666,Betbetbetbetbet6666,Knshshsh,NaN,NaN,NaN,NaN,NaN
6,5176137779,Holla,2019-11-21T05:33:26-05:00,5,1074056860,draku01,Nice app,You make good friends here,NaN,NaN,NaN,NaN,NaN
7,5175821516,Monkey,2019-11-21T03:50:08-05:00,5,967451215,fnfntntj,Sefehtjy,Hthrhrhtjti,NaN,NaN,NaN,NaN,NaN
8,5175755578,Monkey,2019-11-21T03:25:46-05:00,5,1102258972,jasminebadbabg,Baby,I love this app it helps me find the man I need,NaN,NaN,NaN,NaN,NaN
9,5175615226,Monkey,2019-11-21T02:34:08-05:00,5,369541757,Neidnwusk,Aha,Well made,NaN,NaN,NaN,NaN,NaN


---
## Step 2: Filter to the Golden Set

The **golden set** is the subset of reviews that have been manually labeled by a human. You can identify them because the `racism`, `bullying`, and `sexual` columns have actual values (`0` or `1`) rather than being empty.

These are the only rows we can use to evaluate our classifier — we need ground truth labels to calculate precision and recall.

> **Note:** A value of `1` means the review *contains* that type of content (e.g., unwanted sexual comments). A value of `0` means a human reviewed it and decided it does *not*. Empty/NaN means the review was never labeled for that category — which is **different** from `0`.

Since we're building a classifier for sexual content, we filter down further to only rows where the `sexual` column was actually labeled.


In [2]:
# A row is in the golden set if ANY of the label columns has a value.
golden = df[
    df["racism"].notna() | df["bullying"].notna() | df["unwanted_sexual"].notna()
].copy()

print(f"Total rows in dataset:  {len(df)}")
print(f"Rows in golden set:     {len(golden)}")
print(f"\nHow many have each label filled in:")
print(f"  racism labeled:          {golden['racism'].notna().sum()}")
print(f"  bullying labeled:        {golden['bullying'].notna().sum()}")
print(f"  unwanted_sexual labeled: {golden['unwanted_sexual'].notna().sum()}")

# Only keep rows where unwanted_sexual was actually labeled 0 or 1.
# NaN means the review was never labeled and is excluded.
golden_sexual = golden[golden["unwanted_sexual"].isin(["0", "1"])].copy()
golden_sexual["unwanted_sexual"] = golden_sexual["unwanted_sexual"].astype(int)
golden_sexual["review_text"] = golden_sexual["title"].fillna("") + " " + golden_sexual["body"].fillna("")
golden_sexual["review_text"] = golden_sexual["review_text"].str.strip()

# Randomly sample 100 rows to keep API costs low
golden_sexual = golden_sexual.sample(100, random_state=43)

print(f"\nSampled 100 rows from {len(golden[golden['unwanted_sexual'].isin(['0', '1'])])} labeled rows")
print(f"  unwanted_sexual=1 (contains unwanted sexual comments): {golden_sexual['unwanted_sexual'].sum()}")
print(f"  unwanted_sexual=0 (does not):                          {(golden_sexual['unwanted_sexual'] == 0).sum()}")
print(f"\nSample:")
golden_sexual[["review_text", "unwanted_sexual"]].head(5)


Total rows in dataset:  131749
Rows in golden set:     4002

How many have each label filled in:
  racism labeled:          1582
  bullying labeled:        1468
  unwanted_sexual labeled: 3840

Sampled 100 rows from 3323 labeled rows
  unwanted_sexual=1 (contains unwanted sexual comments): 47
  unwanted_sexual=0 (does not):                          53

Sample:


,review_text,unwanted_sexual
131371,@dennis12fowlkes This app has really help me g...,0
34726,Hate It This app has perverts on it BEWARE 😡,1
131417,Loving it! I love this app! I have made a lot ...,0
29850,Skout Good app to get. If your horny the only...,1
131589,Dank Dank,0


---
## Step 3: Write the LLM Prompt

This is the part you'll experiment with! We're going to ask an LLM to read each review and decide whether it contains unwanted sexual content.

The prompt below is a starting point — it's intentionally simple. After you see the precision/recall results, you can come back and tweak the prompt to try to improve the scores.

**Things to experiment with:**
- How specific is your definition of "sexual content"?
- Do you ask for just a yes/no, or ask the model to reason first?
- Do you give examples?
- Do you set a strict or lenient threshold?

We're using [OpenRouter](https://openrouter.ai/) so you can swap in different models without changing the code structure.


In [3]:
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
MODEL = "openai/gpt-5.4-mini"  # ✏️ Change this to try different models

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Using model: {MODEL}")
print(f"API key loaded: {'yes' if OPENROUTER_API_KEY else 'NO - check your .env file'}")


Using model: openai/gpt-5.4-mini
API key loaded: yes


### Structured Outputs

Instead of asking the model to reply with "YES" or "NO" and trying to parse that text, we use **structured outputs** — we give the model a schema and it fills it in like a form.

We define two fields:
- `is_sexual` — a boolean (`True`/`False`)
- `reason` — a short string explaining the decision

The schema is passed directly to the API, so the model is *constrained* to return exactly this shape — not just asked nicely.


In [4]:
class ModerationResult(BaseModel):
    is_sexual: bool  # True if the review contains unwanted sexual comments
    reason: str      # Short explanation of why the model made this decision


### The Prompt

This is the part you'll experiment with! Write a system prompt and a user prompt that together tell the model what to look for.

**Things to try:**
- How specific is your definition of "unwanted sexual comments"?
- Do you ask the model to reason before deciding?
- Do you give examples of what counts (or doesn't count)?


In [5]:
# ✏️ Edit these prompts to experiment!

SYSTEM_PROMPT = """You are a content moderation assistant.
Your job is to detect whether app store reviews contain unwanted sexual comments.
Respond in JSON with two fields: "is_sexual" (boolean) and "reason" (string). 
Only return true if the comment is a compalint about sexual content that is unwanted by the reviewer, not just general discussion of sexual topics."""

USER_PROMPT = """Does the following app store review contain unwanted sexual comments?

Review: {review_text}"""


In [6]:
import json

def classify_sexual_content(review_text: str) -> ModerationResult:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(review_text=review_text)},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ModerationResult",
                "strict": True,
                "schema": ModerationResult.model_json_schema(),
            },
        },
        max_tokens=500,
        temperature=0,
    )
    data = json.loads(response.choices[0].message.content)
    return ModerationResult(**data)


### Sanity Check

Before running on all 331 reviews, test the function on one review to make sure it works.


In [7]:
test_review = "Great app for meeting people!"
result = classify_sexual_content(test_review)

print(f"Review:    {test_review}")
print(f"is_sexual: {result.is_sexual}")
print(f"reason:    {result.reason}")


Review:    Great app for meeting people!
is_sexual: False
reason:    The review is a general positive comment about meeting people and does not mention sexual content or unwanted sexual remarks.


---
## Step 4: Run the Classifier on the Golden Set

Now we run `classify_sexual_content()` on every review in `golden_sexual` and store the results in a new column called `sexual_llm`. You'll see a progress bar as it runs.

> **Note:** We're only running this on rows that were actually labeled for sexual content — that's all we need for evaluation. Running it on all 56k rows would cost money and take a long time, and we don't have ground truth labels for those rows anyway.


In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def _run(idx_and_review):
    idx, review_text = idx_and_review
    for attempt in range(3):
        try:
            result = classify_sexual_content(str(review_text))
            return idx, result.is_sexual, result.reason
        except Exception:
            if attempt == 2:
                raise


Now run it on every review. This sends all the API calls in parallel and shows a progress bar as results come in.


In [9]:
reviews = list(golden_sexual["review_text"].items())
sexual_llm, reasons = {}, {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(_run, item): item for item in reviews}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying reviews"):
        idx, is_sexual, reason = future.result()
        sexual_llm[idx] = int(is_sexual)
        reasons[idx] = reason

golden_sexual["sexual_llm"] = pd.Series(sexual_llm)
golden_sexual["reason"] = pd.Series(reasons)

print(f"LLM flagged as sexual: {golden_sexual['sexual_llm'].sum()}")
print(f"LLM said not sexual:   {(golden_sexual['sexual_llm'] == 0).sum()}")
golden_sexual[["body", "unwanted_sexual", "sexual_llm", "reason"]].head(10)


Classifying reviews:   0%|          | 0/100 [00:00<?, ?it/s]

LLM flagged as sexual: 41
LLM said not sexual:   59


,body,unwanted_sexual,sexual_llm,reason
131371,This app has really help me get a lot of snap ...,0,0,The review mentions getting Snapchat friends a...
34726,This app has perverts on it BEWARE 😡,1,1,The reviewer complains that the app has 'perve...
131417,I love this app! I have made a lot of new frie...,0,0,The review contains a compliment using the wor...
29850,Good app to get. If your horny the only thing...,1,1,The review complains about unwanted sexual con...
131589,Dank,0,0,The review does not mention sexual content or ...
9395,"It’s not user friendly at all,Most people just...",1,1,The review complains that the app is being use...
52293,It's a cool app to meet new people but there's...,1,1,The review complains about encountering 'nasty...
7112,Too many bots messaging me!!!!!\nOver 60% of t...,1,1,The review includes unwanted sexual content co...
129730,There is always some technical issue,1,1,The review includes the sexual insult/complain...
8573,It’s amazing how this app has over 4 stars rat...,0,0,"The review complains about scammers, fake prof..."


---
## Step 5: Calculate Precision and Recall

Now we compare the LLM's predictions (`sexual_llm`) against the human labels (`sexual`) to compute precision and recall.

First, let's understand the four possible outcomes for each review:

| | Human says: Sexual | Human says: Not Sexual |
|---|---|---|
| **LLM says: Sexual** | ✅ True Positive (TP) | ❌ False Positive (FP) |
| **LLM says: Not Sexual** | ❌ False Negative (FN) | ✅ True Negative (TN) |

- **True Positives (TP):** Correctly caught harmful content
- **False Positives (FP):** Flagged innocent reviews (false alarms)  
- **False Negatives (FN):** Missed actual harmful content
- **True Negatives (TN):** Correctly passed innocent reviews

Then:
- **Precision** = TP / (TP + FP) — "How accurate are the flags?"
- **Recall** = TP / (TP + FN) — "How much harmful content did we catch?"


In [10]:
human   = golden_sexual["unwanted_sexual"]  # ground truth (0 or 1)
llm     = golden_sexual["sexual_llm"]       # LLM predictions (0 or 1)

# Count the four outcome types
tp = int(((human == 1) & (llm == 1)).sum())  # both said yes
fp = int(((human == 0) & (llm == 1)).sum())  # LLM said yes, human said no
fn = int(((human == 1) & (llm == 0)).sum())  # LLM said no, human said yes
tn = int(((human == 0) & (llm == 0)).sum())  # both said no

print("Confusion Matrix:")
print(f"  True Positives  (TP): {tp:>4}  — correctly flagged as unwanted sexual comments")
print(f"  False Positives (FP): {fp:>4}  — incorrectly flagged (false alarms)")
print(f"  False Negatives (FN): {fn:>4}  — missed actual unwanted sexual comments")
print(f"  True Negatives  (TN): {tn:>4}  — correctly passed as clean")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n{'='*40}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print(f"{'='*40}")
print(f"\nInterpretation:")
print(f"  - The LLM flagged {tp + fp} reviews as containing unwanted sexual comments")
print(f"  - Of those, {tp} matched the human label ({precision:.0%} precision)")
print(f"  - Out of {tp + fn} reviews humans labeled unwanted_sexual=1, the LLM caught {tp} ({recall:.0%} recall)")


Confusion Matrix:
  True Positives  (TP):   37  — correctly flagged as unwanted sexual comments
  False Positives (FP):    4  — incorrectly flagged (false alarms)
  False Negatives (FN):   10  — missed actual unwanted sexual comments
  True Negatives  (TN):   49  — correctly passed as clean

  Precision: 90.24%
  Recall:    78.72%
  F1 Score:  84.09%

Interpretation:
  - The LLM flagged 41 reviews as containing unwanted sexual comments
  - Of those, 37 matched the human label (90% precision)
  - Out of 47 reviews humans labeled unwanted_sexual=1, the LLM caught 37 (79% recall)


---
## Dig Deeper: Inspect Errors

It's useful to actually *read* the reviews that the model got wrong. Looking at false positives and false negatives often gives you intuition for how to improve the prompt.


In [11]:
pd.set_option("display.max_colwidth", None)

# False Positives: LLM flagged it, but humans said it was fine
false_positives = golden_sexual[(human == 0) & (llm == 1)][["review_text", "unwanted_sexual", "sexual_llm", "reason"]]
print(f"False Positives ({len(false_positives)} reviews the LLM incorrectly flagged):")
false_positives


False Positives (4 reviews the LLM incorrectly flagged):


,review_text,unwanted_sexual,sexual_llm,reason
86493,"Awesome! I love this app, but most of the hot guys are guy..dont waste ur time! (:",0,1,The review mentions sexualized content ('hot guys') in a way that appears to comment on unwanted sexual content in the app.
38436,"Disappointment It has too many fake profiles and a little biased with picture acceptance, several times my pics were removed bc it had a little skin showing but other profiles are just half naked or 1thread away from it still up there... Smh disappointment n dissatisfied.",0,1,The review complains about sexualized/nude profile photos being present on the app and expresses dissatisfaction with that unwanted sexual content.
23265,Craps ppl here This app is now full of crappy ppl .. The gals here r total fake I chatted with 6 gals back to back n all turned out to b asking to view their webcam. And pay by registering to a third party site. Scout Plz filter ur users well .. Not even worth putting any time in this app anymore.,0,1,The review complains about unsolicited sexualized behavior: users asking to view webcams and pay to access a third-party site. It is a complaint about unwanted sexual content/solicitation.
107671,Can you make girls twerk Pls,0,1,"The review requests sexualized content ('girls twerk'), which is an unwanted sexual comment/request."


In [12]:
# False Negatives: Humans flagged it, but LLM missed it
false_negatives = golden_sexual[(human == 1) & (llm == 0)][["review_text", "unwanted_sexual", "sexual_llm", "reason"]]
print(f"False Negatives ({len(false_negatives)} reviews the LLM missed):")
false_negatives


False Negatives (10 reviews the LLM missed):


,review_text,unwanted_sexual,sexual_llm,reason
71837,Eh Too many guts just looking for sex,1,0,"The review mentions sex, but it is a vague complaint and not clearly a complaint about unwanted sexual content in the app."
69355,Perverts Really,1,0,The review is a brief insult and does not explicitly complain about sexual content or contain a sexual comment.
121299,Horny dudes on this app Smh,1,0,The review mentions sexual slang ('horny dudes') but does not clearly complain about unwanted sexual content in the app; it is a general remark rather than a complaint about sexual material.
106910,Fun app to have U just need the filter out dirty men.,1,0,"The review mentions ""dirty men"" and filtering them out, but it does not contain explicit sexual content or a complaint about unwanted sexual content in the app."
87103,Pretty cool! Just gotta watch out for the pervs :) fun to talk to people though!,1,0,"The review mentions 'pervs' in a general cautionary way, but it is not a complaint about unwanted sexual content or explicit sexual remarks."
48271,Full of perverts Everyone on here is perverted.,1,0,"The review complains about people being perverted, but it does not contain sexual content or an unwanted sexual comment itself."
86773,Fun There are some pervs there but overall it's a fun app great way to meet people !!,1,0,"The review mentions ""pervs"" as a general complaint about some users, but it does not contain explicit sexual content or an unwanted sexual comment from the reviewer. It is mainly a general app review."
6620,SKOUT It’s horrible most people if you are a guy there are transexual people not that i have problem with them being who they are but if you guy you understand what I’m saying you care on this app for it to be for either friends or dating not harassment by other people even some of the women on here are creeps!!! Stay away from this app,1,0,"The review mentions transgender people, but it is not complaining about sexual content itself. It is a general complaint about harassment and the app experience, so it does not meet the threshold for unwanted sexual comments."
5937,"Poor App This app is ridiculous! Absolutely ridiculous. It’s filled with trashy, fake people and real people just don’t talk. Nobody talks and people just want money, sex and drugs when they do talk.........of course, why not! That’s what people seem to want than actual friendships and relationships. Plus, this app is so buggy and now I can’t even login into it with Facebook and get error message. I’m done with this app forever!",1,0,"The review mentions sex as part of criticizing the app’s user behavior, but it does not complain about sexual content in the app or express unwanted sexual comments."
74544,it's okay I'm a Lil new to tha app but so far it's not that bad. no bugs. no freezing. but there are a lot of creepy old men!! lol,1,0,"The review mentions 'creepy old men,' but it does not describe sexual content or an unwanted sexual comment. It is a general complaint about users/behavior, not sexual content."
